# Word Embeddings Models
### About this notebook
This notebook is about exploring various embedding models. There are many models, and choosing the right model for the task may be difficult. In this practical you will be exploring static (Word2Vec, Fasttext) as well as dynamic embedding models (BERT).

# Part A. Word Analogy with Embeddings
You will be exploring the use of three different models.
1. Word2Vec Model
2. Fasttext model
3. BERT Model

# Part B. Word similarity with Embeddings.
You will be exploring the use of embeddings from two different models for word similarity calculation.
1. Word2Vec Model
2. BERT Model


# This notebook is about exploring various embedding models. There are many models, and choosing the right model for the task may be difficult. In this practical you will be exploring static (Word2Vec, Fasttext) as well as dynamic embedding models (BERT)

In [ ]:
%%capture
!pip install gensim transformers torch numpy scikit-learn

##Downloading Word2Vec may take 10 minutes. Take this time to review today's lecture on embeddings so that everything is clear for this session, in particular 1) the concept of semantic analogy and 2) semantic similarity with cosine distance

#Part I: Word Analogy with Embeddings

## Part A1. Using word2vec model

In [ ]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

# Download and load a smaller Word2Vec model
word2vec_model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


`Gensim` is a Python library for topic modelling, document indexing and similarity retrieval with large corpora. Target audience is the natural language processing (NLP) and information retrieval (IR) community. You may explore it further at
https://en.wikipedia.org/wiki/Gensim
and
https://pypi.org/project/gensim/

In the above code, you are loading a word2vec model. This word2vec contains pre-trained vectors trained on a part of the Google News dataset (about 100 billion words). The model contains 300-dimensional vectors for 3 million words and phrases.

So, the model `word2vec-google-news-300` transforms words into 300-dimensional vector representations. The model contains vector representations for 3 million words and common phrases, enabling semantic analysis of text through mathematical operations on word vectors.


**Model Inputs and Outputs:**

Inputs: Words or phrases from a vocabulary of 3 million terms, Text requiring semantic analysis or comparison

Outputs: 300-dimensional vectors representing words and phrases, Vector spaces that capture semantic relationships between terms.

`KeyedVectors` is essentially a mapping between keys and vectors. Each vector is identified by its lookup key, most often a short string token, so this is usually a mapping between {str => 1D numpy array}.




Let's define word analogy based on existing Word2Vec functions and test it first with the "textbook" analogy you have also seen in the lecture.

In [ ]:
def word_analogy(model, positive, negative, topn=5):
    """
    Finds words that fit the analogy: positive1 - negative1 + positive2 = ?
    :param model: Word2Vec model
    :param positive: List of positive words (e.g., ['king', 'woman'])
    :param negative: List of negative words (e.g., ['man'])
    :param topn: Number of closest results to return
    """
    return model.most_similar(positive=positive, negative=negative, topn=topn)

In [ ]:
# Example analogy: king - man + woman = queen
result = word_analogy(word2vec_model, positive=['king', 'woman'], negative=['man'])
print("Word Analogy Results:", result)

Word Analogy Results: [('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581)]


Let's define more analogies. The format is:

*   positive = two words for instance 'king' and 'woman'
*   negative = one word, which is associated to the first word, to be "subtracted" and replace by the second positive word, for instance 'man', which is associated to 'king' and will be replaced by 'woman' with an expected result of 'queen'



In [ ]:
analogies = [
    (['king', 'woman'], ['man']),  # king - man + woman = queen
    (['paris', 'italy'], ['france']),  # paris - france + italy = rome
    (['walking', 'swam'], ['walked']),  # walking - walked + swam = swum
    (['big', 'bigger'], ['small']),  # big - small + bigger = smaller
    (['doctor', 'woman'], ['man']),  # doctor - man + woman = nurse (semantic bias example)
]

In [ ]:
for positive, negative in analogies:
    result = word_analogy(word2vec_model, positive, negative)
    print(f"{positive} - {negative} = {result[0][0]} (Score: {result[0][1]:.4f})")

['king', 'woman'] - ['man'] = queen (Score: 0.7118)
['paris', 'italy'] - ['france'] = lohan (Score: 0.5070)
['walking', 'swam'] - ['walked'] = swimming (Score: 0.7449)
['big', 'bigger'] - ['small'] = biggest (Score: 0.5590)
['doctor', 'woman'] - ['man'] = gynecologist (Score: 0.7094)


Let's try with another set. Record those cases where the analogy works and those where it doesn't. When it doesn't work it may not be because the analogy is wrong but because limitations of the model (Word2Vec is rather old now, and is a static model).
You can also try with your own examples!

In [ ]:
analogies = [
    (['king', 'woman'], ['man']),       # king - man + woman = queen
    (['paris', 'france'], ['london']),  # paris - france + london = england
    (['japan', 'sushi'], ['italy']),    # japan - sushi + italy = pizza
    (['fast', 'faster'], ['slow']),     # fast - slow + faster = slower
]

In [ ]:
for positive, negative in analogies:
    result = word_analogy(word2vec_model, positive, negative)
    print(f"{positive} - {negative} = {result[0][0]} (Score: {result[0][1]:.4f})")

['king', 'woman'] - ['man'] = queen (Score: 0.7118)
['paris', 'france'] - ['london'] = France (Score: 0.5404)
['japan', 'sushi'] - ['italy'] = Sushi (Score: 0.5640)
['fast', 'faster'] - ['slow'] = quicker (Score: 0.5878)


## Part A2. Using Fasttext model

##Let's try with a more recent embedding model, Fasttext. It is still a static embedding model, so it doesn't account for the different meanings of a word based on context. But it handles out-of-vocabulary words better than Word2Vec and captures morphological nuances (e.g., prefixes and suffixes).

In [ ]:
%%capture
!pip install fasttext

We'lle be downloading the .vec version of Fasttext (you'll use the .bin version later).
*   Text-based: The .vec file is a plain text file where each line represents a word and its corresponding vector.
*   Human-readable: You can open .vec files in a text editor and read the word embeddings directly. Each line contains the word followed by the vector values (usually space-separated).




In [ ]:
# Takes 2-3 minutes
# Download the English FastText model
# There is one embedding per language
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
!gunzip cc.en.300.vec.gz

--2026-03-20 18:18:39--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.88, 54.230.79.53, 54.230.79.93, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.88|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G  17.1MB/s    in 12s     

2026-03-20 18:18:51 (107 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]



##Below we are loading under the word2vec format to support the same functions for computing analogy between words.
##This akes another 10 minutes... so maybe take this time to collect i) more examples of analogies, and ii) also examples of word pairs where the words have different degrees of similarity.

In [ ]:
from gensim.models import KeyedVectors

# Load the pre-trained FastText embeddings
fasttext_model_path = "/content/cc.en.300.vec"  # Update with your file
fasttext_model = KeyedVectors.load_word2vec_format(fasttext_model_path, binary=False)   #for analogy tasks

print(f"Loaded {len(fasttext_model)} word vectors.")

Loaded 2000000 word vectors.


##Now we are defining an analogy function just as before but with Fasttext

In [ ]:
# Solve analogies using FastText
def solve_analogy(model, positive, negative):
    try:
        result = model.most_similar(positive=positive, negative=negative, topn=1)
        return result
    except KeyError as e:
        return f"Word not in vocabulary: {e}"

In [ ]:
# Example analogies
analogies = [
    (['king', 'woman'], ['man']),
    (['paris', 'italy'], ['france']),
    (['walking', 'swam'], ['walked']),
    (['japan', 'sushi'], ['italy']),
    (['doctor', 'woman'], ['man']),
]

#### Test the `solve_anaogy` function for the above analogies.

In [ ]:
for positive, negative in analogies:
    result = solve_analogy(fasttext_model, positive, negative)
    print(f"{positive} - {negative} = {result}")

['king', 'woman'] - ['man'] = [('queen', 0.7554903030395508)]
['paris', 'italy'] - ['france'] = [('rome', 0.6593306064605713)]
['walking', 'swam'] - ['walked'] = [('swimming', 0.7572133541107178)]
['japan', 'sushi'] - ['italy'] = [('sashimi', 0.6613127589225769)]
['doctor', 'woman'] - ['man'] = [('gynecologist', 0.7013276219367981)]


###look at which analogies have improved and which have not. Also try with your own examples of analogies.

## Part A3. Using BERT model


# Now let's use a contextual embedding, BERT.
### BERT stands for Bidirectional Encoder Representations from Transformers. It is a language model developed by Google that improves the understanding of text by analyzing the context of words in both directions. It is used for various natural language processing tasks, such as sentiment analysis, question answering, and text summarization.

### We no longer have the Word2Vec format that makes it possible to define an analogy function, so we will measure the similarity between the expected results and the analogy equation of the type (king + woman - man). The higher the similarity the closer we are to the results. You can compare to previous wrong results by measuring the distance to the wrong results obtained e.g. with Word2Vec.

In [ ]:
# IGNORE the warning about the HF_TOKEN as we're using public models
from transformers import AutoModel, AutoTokenizer
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
bert_model = AutoModel.from_pretrained(model_name)
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Embed words
def embed_word(word):
    inputs = bert_tokenizer(word, return_tensors="pt")
    outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

In [ ]:
# Perform analogy: doctor - man + woman
doctor = embed_word("doctor")
man = embed_word("man")
woman = embed_word("woman")
nurse = embed_word("nurse")

result_vector = doctor - man + woman

# Compare to queen
similarity = cosine_similarity(result_vector, nurse)
print(f"Similarity to 'nurse': {similarity[0][0]:.4f}")

In [ ]:
# Perform analogy: doctor - man + woman
japan = embed_word("Japan")
sushi = embed_word("sushi")
pizza = embed_word("pizza")
italy = embed_word("Italy")

result_vector = italy - sushi + japan

# Compare to queen
similarity = cosine_similarity(result_vector, pizza)
print(f"Similarity to 'pizza': {similarity[0][0]:.4f}")

#Part B: Word similarity with embeddings

In [ ]:
word_pairs = [
    ("king", "queen"),
    ("cat", "dog"),
    ("car", "bicycle"),
    ("apple", "banana"),
    ("teacher", "school"),
    ("war", "peace"),  # antonyms
    ("happy", "joyful"),
    ("table", "blue"), # unrelated
]

## Part B1. Using Word2Vec Model for word similarity.

Note: As you have already loaded the word2vec model as `word2vec_model`, you can use it here.

In [ ]:
def word2vec_similarity(word1, word2):
    try:
        vec1 = word2vec_model[word1]
        vec2 = word2vec_model[word2]
        return cosine_similarity([vec1], [vec2])[0][0]
    except KeyError:
        return None  # Handle OOV words

# Compute similarities
for word1, word2 in word_pairs:
    similarity = word2vec_similarity(word1, word2)
    print(f"Word2Vec Similarity ({word1}, {word2}): {similarity}")

## Part B2. Using BERT Model for word similarity

In [ ]:
# IGNORE the warning about the HF_TOKEN as we're using public models
# Reloading with a slight variant
from transformers import AutoModel, AutoTokenizer
import torch

# Load BERT model and tokenizer
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def bert_embedding(word):
    tokens = bert_tokenizer(word, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        output = bert_model(**tokens)
    # Average the token embeddings
    return output.last_hidden_state.mean(dim=1).squeeze().numpy()

###Let's compute BERT similarity. Compare to the Word2Vec scores.
###You can try more word pairs of your own

In [ ]:
def bert_similarity(word1, word2):
    vec1 = bert_embedding(word1)
    vec2 = bert_embedding(word2)
    return cosine_similarity([vec1], [vec2])[0][0]

# Compute similarities
for word1, word2 in word_pairs:
    similarity = bert_similarity(word1, word2)
    print(f"BERT Similarity ({word1}, {word2}): {similarity}")



---



##Let's move from word to sentences and compute sentence similarity

In [ ]:
sentence_pairs = [
    ("The king rules the kingdom.", "The queen governs the empire."),
    ("The cat is on the mat.", "The dog is in the yard."),
    ("I love playing football.", "Soccer is my favorite sport."),
    ("War brings destruction.", "Peace leads to harmony."),
    ("The teacher is in the classroom.", "The student is in the library."),
    ("I love programming.", "I enjoy coding."),
    ("The weather is nice today.", "It's sunny outside."),
    ("Apples are delicious.", "I like bananas.")
]

### Using Word2Vec for sentence similarity

In [ ]:
def sentence_embedding_word2vec(sentence):
    words = sentence.lower().split()
    vectors = [word2vec_model[word] for word in words if word in word2vec_model]
    return sum(vectors) / len(vectors) if vectors else None

def word2vec_sentence_similarity(sent1, sent2):
    vec1 = sentence_embedding_word2vec(sent1)
    vec2 = sentence_embedding_word2vec(sent2)
    if vec1 is not None and vec2 is not None:
        return cosine_similarity([vec1], [vec2])[0][0]
    return None

# Compute similarities
for sent1, sent2 in sentence_pairs:
    similarity = word2vec_sentence_similarity(sent1, sent2)
    print(f"Word2Vec Sentence Similarity:\n{sent1}\n{sent2}\nScore: {similarity}\n")

###Then, using BERT for sentence similarity

In [ ]:
def sentence_embedding_bert(sentence):
    tokens = bert_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        output = bert_model(**tokens)
    # Average over all token embeddings
    return output.last_hidden_state.mean(dim=1).squeeze().numpy()

def bert_sentence_similarity(sent1, sent2):
    vec1 = sentence_embedding_bert(sent1)
    vec2 = sentence_embedding_bert(sent2)
    return cosine_similarity([vec1], [vec2])[0][0]

# Compute similarities
for sent1, sent2 in sentence_pairs:
    similarity = bert_sentence_similarity(sent1, sent2)
    print(f"BERT Sentence Similarity:\n{sent1}\n{sent2}\nScore: {similarity}\n")

##Now analyse the results. Try with different sentence pairs, possibly longer ones.

### So BERT is contextual, while Word2Vec and Fasttext focus on isolated words
### Now, let's try SBERT which is optimised for sentence comparison.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the SBERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # Or another SBERT model of your choice

# Function to compute SBERT embeddings for a sentence
def sentence_embedding_sbert(sentence):
    return sbert_model.encode(sentence, convert_to_numpy=True)

# Function to compute cosine similarity between two sentences using SBERT
def sbert_sentence_similarity(sent1, sent2):
    vec1 = sentence_embedding_sbert(sent1)
    vec2 = sentence_embedding_sbert(sent2)
    return cosine_similarity([vec1], [vec2])[0][0]

In [ ]:
for sent1, sent2 in sentence_pairs:
    similarity = sbert_sentence_similarity(sent1, sent2)
    print(f"SBERT Sentence Similarity:\n{sent1}\n{sent2}\nScore: {similarity}\n")

## Analyse the results. Now try with longer sentences to see if SBERT does better and in which case.